# SAM2.1 Field Boundary Prediction -- SGA Test Images

Uses local weights weights/sam2.1_hiera_large.pt.

| Sensor | Strategy |
|---|---|
| Drone (~1600x1480 px) | Resize longest side to 1024 px, single pass |
| Satellite (~3250x3266 px) | 1024x1024 patches with 128 px overlap, merge |

**Pipeline:** SAM2 inference -> morphological opening -> watershed -> polygonize -> overlay

In [ ]:
import sys, os, contextlib
import pandas as pd

# SAM2 repo path
_SAM2_REPO = os.path.normpath(os.path.join(os.path.abspath('..'), 'sam2'))
if _SAM2_REPO not in sys.path:
    sys.path.insert(0, _SAM2_REPO)
sys.path.insert(0, os.path.abspath('.'))   # ensure notebooks/ is importable

from sam_utils import *                    # helpers, plot funcs, pipeline funcs

import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
print(f'device : {DEVICE}')

In [ ]:
REPO_ROOT     = os.path.abspath('..')
TEST_DATA_DIR = os.path.join(REPO_ROOT, 'data', 'sga-test')
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'outputs', 'sam2_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAM2_CKPT   = os.path.join(REPO_ROOT, 'weights', 'sam2.1_hiera_large.pt')
SAM2_CONFIG = 'configs/sam2.1/sam2.1_hiera_l.yaml'

SGA_GROUPS = {
    'drone':     {'dir': os.path.join(TEST_DATA_DIR, 'drone'),
                  'files': ['C081_orginal.jpg', 'H6_orginal.jpg', 'S10_orginal.jpg']},
    'satellite': {'dir': os.path.join(TEST_DATA_DIR, 'satellite'),
                  'files': ['demo_area01.jpg', 'demo_area02.jpg']},
}

# Inference
DRONE_MAX_SIDE      = 1024
SAT_PATCH_SIZE      = 1024
SAT_PATCH_OVERLAP   = 128
DRONE_GRID          = 12
SAT_PATCH_GRID      = 8
IOU_DEDUP_THRESHOLD = 0.7
MIN_MASK_AREA_RATIO = 0.005
BOUNDARY_DILATION   = 2
MIN_SCORE_THRESHOLD_DRONE = 0.95
MIN_SCORE_THRESHOLD_SATELLITE = 0.85

# Post-processing
MORPH_KERNEL  = 2    # disk radius for morphological opening
WSHED_KERNEL  = 5    # min_distance for watershed peak detection
POLY_MIN_SIZE = 500  # m² — minimum polygon area to keep

# Dummy geospatial parameters (EPSG:32632 UTM, 10 m pixels)
PIXEL_SIZE_M = 10.0
ORIGIN_X     = 500_000.0
ORIGIN_Y     = 5_400_000.0
TARGET_CRS   = rasterio.crs.CRS.from_epsg(32632)

print(f'Checkpoint : {SAM2_CKPT}')
print(f'Output dir : {OUTPUT_DIR}')

## 1. Load SGA Test Images

In [ ]:
images = {}

for group_name, info in SGA_GROUPS.items():
    for fname in info['files']:
        stem = os.path.splitext(fname)[0]
        path = os.path.join(info['dir'], fname)
        rgb  = np.array(Image.open(path).convert('RGB'))
        images[stem] = {'rgb': rgb, 'group': group_name, 'path': path}
        h, w = rgb.shape[:2]
        print(f'  [{group_name:9s}] {fname}  {h}x{w} px')

print(f'Loaded {len(images)} images.')

## 2. Load SAM2.1 (local weights)

Loads weights/sam2.1_hiera_large.pt via uild_sam2 -- no internet needed.

In [ ]:
sam_model = build_sam2(SAM2_CONFIG, ckpt_path=SAM2_CKPT, device=DEVICE)
predictor = SAM2ImagePredictor(sam_model)
print(f'SAM2.1 hiera-large loaded from local weights  (device: {DEVICE})')

## 4. Inference

| Sensor | Pre-process | SAM2 strategy |
|---|---|---|
| Drone | Resize longest side to 1024 px | Single pass, 12x12 grid |
| Satellite | Native resolution | 1024x1024 patches, 8x8 grid each |


In [ ]:
results = {}

autocast_ctx = (lambda: torch.autocast('cuda', dtype=torch.bfloat16)
                if DEVICE == 'cuda' else contextlib.nullcontext)

with torch.inference_mode(), autocast_ctx():
    for stem, info in images.items():
        rgb_orig = info['rgb']
        group    = info['group']
        h0, w0   = rgb_orig.shape[:2]

        if group == 'drone':
            rgb_inf = resize_keep_aspect(rgb_orig, DRONE_MAX_SIDE)
            h, w    = rgb_inf.shape[:2]
            print(f'\n[drone    ] {stem}  {h0}x{w0} -> {h}x{w}  grid={DRONE_GRID}x{DRONE_GRID}')
            kept                = predict_single(predictor, rgb_inf, DRONE_GRID,
                                                 MIN_MASK_AREA_RATIO, IOU_DEDUP_THRESHOLD,
                                                 MIN_SCORE_THRESHOLD_DRONE)
            semantic, score_map = masks_to_semantic(kept, h, w, BOUNDARY_DILATION)
            display_rgb = rgb_inf
            print(f'  fields detected: {len(kept)}')
        else:
            rgb_inf = rgb_orig
            h, w    = rgb_inf.shape[:2]
            print(f'\n[satellite] {stem}  {h}x{w}  '
                  f'patch={SAT_PATCH_SIZE} overlap={SAT_PATCH_OVERLAP}  '
                  f'grid={SAT_PATCH_GRID}x{SAT_PATCH_GRID}')
            semantic, score_map = predict_satellite_patches(
                predictor, rgb_inf,
                SAT_PATCH_SIZE, SAT_PATCH_OVERLAP, SAT_PATCH_GRID,
                MIN_MASK_AREA_RATIO, IOU_DEDUP_THRESHOLD, BOUNDARY_DILATION,
                MIN_SCORE_THRESHOLD_SATELLITE,
            )
            display_rgb = rgb_inf

        pred_tif  = os.path.join(OUTPUT_DIR, f'{stem}_pred.tif')
        score_tif = os.path.join(OUTPUT_DIR, f'{stem}_score.tif')
        save_semantic_geotiff(semantic, pred_tif, ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M, TARGET_CRS)
        save_score_geotiff(score_map, score_tif, ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M, TARGET_CRS)
        Image.fromarray(semantic_to_rgb(semantic)).save(
            os.path.join(OUTPUT_DIR, f'{stem}_sam2_semantic.png')
        )

        results[stem] = dict(semantic=semantic, score_map=score_map,
                             display_rgb=display_rgb, pred_tif=pred_tif, score_tif=score_tif)
        print(f'  Saved: {os.path.basename(pred_tif)}')

print('\nAll images processed.')

## 5. Raw SAM2 Predictions

Original RGB | semantic mask | colour overlay

In [ ]:
plot_sam2_predictions(results, images, OUTPUT_DIR)

## 6. Post-Processing

1. **Morphological opening** (`MORPH_KERNEL`) — removes small noise blobs from the field class.
2. **Watershed segmentation** (`WSHED_KERNEL`) — splits touching field instances into labelled regions.

In [ ]:
morph_tif_paths, wshed_tif_paths, postproc_arrs = run_postprocessing(
    results, OUTPUT_DIR, MORPH_KERNEL, WSHED_KERNEL
)
plot_postprocessing(results, postproc_arrs, OUTPUT_DIR, MORPH_KERNEL, WSHED_KERNEL)

## 7. Polygonisation

Converts the morphological prediction TIF (class labels 0/1/2) to vector polygons
using rasterio.features.shapes directly.

**Parameters:**
- POLY_MIN_SIZE=500 m2 -- drop polygons smaller than 500 m2 (5 pixels at 10 m)
- Dummy EPSG:32632 CRS; polygon coordinates and areas have no real-world meaning.

In [ ]:
polygon_results = run_polygonize(morph_tif_paths, OUTPUT_DIR, POLY_MIN_SIZE, PIXEL_SIZE_M)

## 8. Polygon Overlay on Original Image

Field polygons (from morphological prediction) drawn over the original RGB.
Green fill = field interior, red edge = boundary.

In [ ]:
plot_polygon_overlays(polygon_results, results, images, ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M, OUTPUT_DIR)

## 9. Per-Image Statistics

In [ ]:
rows = []
for stem, res in results.items():
    sem   = res['semantic']
    total = sem.size
    h, w  = sem.shape
    gdf   = polygon_results.get(stem)
    rows.append({
        'Image':            stem,
        'Group':            images[stem]['group'],
        'H x W (inferred)': f'{h}x{w}',
        'Field px %':       f'{100*(sem==1).sum()/total:.1f}',
        'Boundary px %':    f'{100*(sem==2).sum()/total:.1f}',
        'Polygons':         len(gdf) if gdf is not None else 0,
    })
display(pd.DataFrame(rows))

## 11. Interactive HTML Viewer

One self-contained HTML file per image saved to `outputs/sam2_results/`.

- Toggle polygon overlay on/off with the button
- Click any polygon to see its **area** (px² and m²) and **SAM2 confidence score**
- Confidence is the mean score of SAM2 masks that cover the polygon (populated after re-running inference; shows N/A when score TIF is absent)

In [ ]:
for stem, gdf in polygon_results.items():
    path = make_interactive_html(
        stem, results[stem]['display_rgb'], gdf, OUTPUT_DIR,
        images[stem]['group'], ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M,
    )
    print(f'  {os.path.basename(path)}  ({len(gdf)} polygons)')

print(f'\nOpen any *_interactive.html in {OUTPUT_DIR}')